# YOLO Fire Detector - Persistent Cloud Training

Notebook operativo per eseguire in cloud l'intera pipeline del progetto: sync del repository, generazione o riuso del dataset, training YOLOv8, registrazione dell'export finale e creazione dello zip da riportare in locale.

## Prima di iniziare

1. Da VS Code locale prepara il bundle con `python tools/cloud/prepare_cloud_bundle.py`.
2. Verifica che esista `dist/yolo-fire-detector-cloud.zip`.
3. Apri questo notebook in Colab.
4. Se vuoi usare la GPU, abilitala in `Runtime -> Change runtime type`.
5. Esegui le celle in ordine, senza lanciare separatamente `generator.py` o `train.py`.

## Preset consigliato

Per partire senza complicarti il flusso, usa il preset:

- `cloud.balanced-mini-fires.yaml`

Questo preset usa gli asset `fire1-mini_nobg.png` e `fire2-mini_nobg.png` ed e' pensato come compromesso tra tempo, costo e qualita'.

## Cosa fa questo notebook

Questo notebook usa una root persistente unica e mantiene nello stesso punto:

- codice del progetto
- dataset generati e relativi manifest YAML
- run di training e diagnostica YOLO
- export finali e registry degli export

Su Colab monta Google Drive e riparte automaticamente dallo stesso stato se esiste gia' un dataset compatibile o una run riprendibile.

## Root persistente e bundle

Il notebook cerca automaticamente `yolo-fire-detector-cloud.zip` e lo puo' trovare in uno di questi posti:

- `/content`
- Google Drive

Se trova il file fuori dalla root persistente, lo sposta in `inputs/` e aggiorna la copia del repository in `repo/`.

In [ ]:
import json

import os

from pathlib import Path

import shutil

import sys

import zipfile


IN_COLAB = 'google.colab' in sys.modules

BUNDLE_NAME = 'yolo-fire-detector-cloud.zip'


def find_local_notebook_root() -> Path:
    current = Path.cwd().resolve()
    fallback = current

    for candidate in [current, *current.parents]:
        has_project_files = (candidate / 'cloud_train.ipynb').exists() and (candidate / 'run_experiment.py').exists()
        if has_project_files:
            fallback = candidate
        if has_project_files and (candidate / 'dist' / BUNDLE_NAME).exists():
            return candidate

    return fallback


if IN_COLAB:

    from google.colab import drive

    drive.mount('/content/drive')

    LOCAL_NOTEBOOK_ROOT = None
    PERSISTENT_ROOT = Path('/content/drive/MyDrive/yolo-fire-detector')

else:

    LOCAL_NOTEBOOK_ROOT = find_local_notebook_root()
    PERSISTENT_ROOT = LOCAL_NOTEBOOK_ROOT / 'artifacts' / 'cloud-notebook-local'


REPO_ROOT = PERSISTENT_ROOT / 'repo'

INPUTS_ROOT = PERSISTENT_ROOT / 'inputs'

RUNTIME_CONFIG_PATH = REPO_ROOT / 'configs' / 'cloud.runtime.yaml'


PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

INPUTS_ROOT.mkdir(parents=True, exist_ok=True)


print('IN_COLAB =', IN_COLAB)

print('BUNDLE_NAME =', BUNDLE_NAME)

print('LOCAL_NOTEBOOK_ROOT =', LOCAL_NOTEBOOK_ROOT)

print('PERSISTENT_ROOT =', PERSISTENT_ROOT)

print('REPO_ROOT =', REPO_ROOT)

print('INPUTS_ROOT =', INPUTS_ROOT)

print('RUNTIME_CONFIG_PATH =', RUNTIME_CONFIG_PATH)

In [ ]:
PURGE_PERSISTENT_REPO = False
PURGE_PERSISTENT_INPUTS_CACHE = False
PURGE_PERSISTENT_OUTPUTS = False


def remove_tree_if_exists(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)
        print('Rimosso:', path)
    else:
        print('Gia assente:', path)


if PURGE_PERSISTENT_REPO:
    remove_tree_if_exists(REPO_ROOT)

if PURGE_PERSISTENT_INPUTS_CACHE:
    remove_tree_if_exists(INPUTS_ROOT)
    INPUTS_ROOT.mkdir(parents=True, exist_ok=True)
    print('Ricreata cache inputs:', INPUTS_ROOT)

if PURGE_PERSISTENT_OUTPUTS:
    remove_tree_if_exists(PERSISTENT_ROOT / 'datasets')
    remove_tree_if_exists(PERSISTENT_ROOT / 'runs')
    remove_tree_if_exists(PERSISTENT_ROOT / 'exports')
    remove_tree_if_exists(PERSISTENT_ROOT / 'downloads')

print('PURGE_PERSISTENT_REPO =', PURGE_PERSISTENT_REPO)
print('PURGE_PERSISTENT_INPUTS_CACHE =', PURGE_PERSISTENT_INPUTS_CACHE)
print('PURGE_PERSISTENT_OUTPUTS =', PURGE_PERSISTENT_OUTPUTS)
print('Workspace persistente pronta per il sync.')

## Sync progetto nella root persistente

Questa sezione prepara il contesto di lavoro usato da tutte le celle successive.

Prima della cella di sync trovi una cella separata con tre interruttori opzionali:

- `PURGE_PERSISTENT_REPO`: cancella la copia estratta in `repo/`
- `PURGE_PERSISTENT_INPUTS_CACHE`: cancella la cache locale del bundle in `inputs/`
- `PURGE_PERSISTENT_OUTPUTS`: cancella dataset, run, export e download persistenti

Usa almeno `PURGE_PERSISTENT_REPO = True` se vuoi essere certo che il notebook rigeneri il codice partendo davvero dallo zip invece di riusare una copia vecchia presente in Google Drive.

La cella qui sotto fa automaticamente questo:

1. controlla se il bundle `yolo-fire-detector-cloud.zip` e' gia' presente in `inputs/`
2. se non c'e', lo cerca nel contesto locale della sessione e in Google Drive
3. se lo trova fuori dalla root persistente, lo copia in `<PERSISTENT_ROOT>/inputs/`
4. estrae o aggiorna il repository in `<PERSISTENT_ROOT>/repo/`

Percorsi principali usati dal notebook:

- `PERSISTENT_ROOT`: root persistente del lavoro
- `INPUTS_ROOT`: cartella dove viene conservato lo zip del progetto
- `REPO_ROOT`: cartella del repository estratto
- `RUNTIME_CONFIG_PATH`: file `configs/cloud.runtime.yaml` generato dal notebook

### Quando intervenire manualmente

- Se vuoi forzare l'aggiornamento del codice estratto, imposta `FORCE_PROJECT_REFRESH = True`.
- Se il notebook non trova lo zip, metti `yolo-fire-detector-cloud.zip` in `/content` o in Google Drive e riesegui questa sezione.
- Se temi codice vecchio inquinante, usa la cella di purge prima della sync.

In [ ]:
FORCE_PROJECT_REFRESH = False


def find_bundle_candidates(bundle_name: str) -> list[Path]:
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        if path.exists() and path not in candidates:
            candidates.append(path)

    add_candidate(INPUTS_ROOT / bundle_name)

    local_search_roots = [Path.cwd()]
    if not IN_COLAB and LOCAL_NOTEBOOK_ROOT is not None:
        local_search_roots.extend(
            [
                LOCAL_NOTEBOOK_ROOT,
                LOCAL_NOTEBOOK_ROOT / 'dist',
                LOCAL_NOTEBOOK_ROOT.parent,
                LOCAL_NOTEBOOK_ROOT.parent / 'dist',
            ]
        )
    else:
        local_search_roots.append(Path('/content'))

    for root in local_search_roots:
        if not root.exists():
            continue
        add_candidate(root / bundle_name)

    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        if drive_root.exists():
            add_candidate(drive_root / bundle_name)
            for match in drive_root.rglob(bundle_name):
                add_candidate(match)

    return candidates


def ensure_bundle_in_persistent_inputs(bundle_name: str) -> Path | None:
    candidates = find_bundle_candidates(bundle_name)
    if not candidates:
        return None

    chosen_bundle = candidates[0]
    persistent_bundle = INPUTS_ROOT / bundle_name

    if chosen_bundle.resolve() != persistent_bundle.resolve():
        INPUTS_ROOT.mkdir(parents=True, exist_ok=True)
        if persistent_bundle.exists():
            persistent_bundle.unlink()
        shutil.copy2(str(chosen_bundle), str(persistent_bundle))
        print('Bundle copiato in root persistente:', persistent_bundle)
    else:
        print('Bundle gia presente in root persistente:', persistent_bundle)

    return persistent_bundle


bundle_path = ensure_bundle_in_persistent_inputs(BUNDLE_NAME)
repo_ready = (REPO_ROOT / 'run_experiment.py').exists()


if bundle_path is None and not repo_ready:
    raise FileNotFoundError(
        'Bundle non trovato. Metti yolo-fire-detector-cloud.zip in dist/, nella root del repo, in /content o in Google Drive e rilancia la cella.'
    )


if bundle_path is not None and (FORCE_PROJECT_REFRESH or not repo_ready):
    temp_extract_dir = PERSISTENT_ROOT / '_incoming_repo'

    shutil.rmtree(temp_extract_dir, ignore_errors=True)
    shutil.rmtree(REPO_ROOT, ignore_errors=True)

    temp_extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(bundle_path, 'r') as archive:
        archive.extractall(temp_extract_dir)

    extracted_files = [child for child in temp_extract_dir.iterdir() if child.is_file()]
    extracted_roots = [child for child in temp_extract_dir.iterdir() if child.is_dir()]
    if extracted_files:
        source_root = temp_extract_dir
    elif len(extracted_roots) == 1:
        source_root = extracted_roots[0]
    else:
        source_root = temp_extract_dir

    shutil.copytree(source_root, REPO_ROOT, dirs_exist_ok=True)
    shutil.rmtree(temp_extract_dir, ignore_errors=True)


print('Bundle persistente =', bundle_path)
print('Repo pronta =', REPO_ROOT.exists())
print('run_experiment.py presente =', (REPO_ROOT / 'run_experiment.py').exists())

## Installazione dipendenze e ambiente di esecuzione

Dopo che il repository e' stato sincronizzato, la cella successiva:

1. entra nella cartella `repo/` estratta dal bundle
2. installa le dipendenze da `requirements.txt`
3. prova a mostrare lo stato GPU con `nvidia-smi`

Qui usiamo comandi notebook diretti invece di `subprocess`, cosi' l'output resta visibile in modo piu' completo e il comportamento e' piu' naturale in Colab.

Se la sessione e' gia' pronta e le dipendenze sono gia' installate, puoi comunque rieseguire la cella senza cambiare il flusso del notebook.

In [ ]:
import os

from shutil import which

os.chdir(REPO_ROOT)
print('Working directory impostata su', REPO_ROOT)

get_ipython().run_line_magic('cd', str(REPO_ROOT))
get_ipython().system(f'"{sys.executable}" -m pip install -r requirements.txt')

if which('nvidia-smi'):
    get_ipython().system('nvidia-smi')
else:
    print('nvidia-smi non disponibile in questa sessione; controllo GPU di sistema saltato.')

In [ ]:
import os

import torch

CUDA_READY = torch.cuda.is_available()
CUDA_DEVICE_COUNT = torch.cuda.device_count()
CUDA_VISIBLE_DEVICES = os.environ.get('CUDA_VISIBLE_DEVICES')
TORCH_VERSION = torch.__version__
TORCH_CUDA_VERSION = torch.version.cuda

print('Torch version =', TORCH_VERSION)
print('Torch CUDA build =', TORCH_CUDA_VERSION)
print('torch.cuda.is_available() =', CUDA_READY)
print('torch.cuda.device_count() =', CUDA_DEVICE_COUNT)
print("CUDA_VISIBLE_DEVICES =", CUDA_VISIBLE_DEVICES)

if CUDA_READY:
    print('GPU torch visibile =', torch.cuda.get_device_name(0))
else:
    print('ATTENZIONE: la runtime puo avere GPU Colab assegnata, ma PyTorch in questa sessione non la vede.')
    print('Cause tipiche:')
    print('- runtime non realmente collegata alla GPU dopo il cambio hardware')
    print('- kernel riusato o non riavviato dopo il cambio runtime')
    print('- build di torch CPU-only nella sessione corrente')
    print('Se vuoi proseguire senza blocco, il notebook usera CPU automaticamente.')

## Configurazione runtime

Questa e' la parte in cui scegli come costruire `repo/configs/cloud.runtime.yaml`. Hai due modalita'.

### Modalita' A: usa una config gia' pronta

Imposta:

- `USE_READY_CONFIG_AS_IS = True`
- `READY_CONFIG_NAME = 'cloud.balanced-mini-fires.yaml'`

In questo caso il notebook usa direttamente quel preset e cambia solo `project.persistent_root`.

### Modalita' B: parti da un preset e sovrascrivi i campi principali

Imposta:

- `USE_READY_CONFIG_AS_IS = False`
- `BASE_CONFIG_NAME = 'cloud.balanced-mini-fires.yaml'`

Poi puoi modificare i parametri principali:

- `PROJECT_LABEL`
- `DATASET_LABEL`, `FIRE_IMAGE_PATHS`, `NUM_IMAGES`, `IMAGE_SIZE`, `NEGATIVE_RATIO`, `TRAIN_SPLIT`, `DATASET_SEED`, `FORCE_REGENERATE`
- `TRAINING_LABEL`, `MODEL_SIZE`, `WEIGHTS`, `DEVICE`, `EPOCHS`, `BATCH_SIZE`, `RESUME_POLICY`
- `DATASET_SETTINGS_OVERRIDES`, `IMAGE_TRANSFORM_OVERRIDES`, `TRAINING_OVERRIDES`

### Patch manuale completa

La cella successiva ti permette di applicare `MANUAL_CONFIG_PATCH` a qualunque chiave annidata della config prima del salvataggio finale di `cloud.runtime.yaml`.

### Note pratiche

- Se vuoi rigenerare il dataset, imposta `dataset.force_regenerate: true`.
- Se vuoi evitare il resume, usa `training.resume: never`.
- Se vuoi semplicemente lanciare un preset senza modifiche, resta in Modalita' A.

In [ ]:
import yaml


AVAILABLE_BASE_CONFIGS = [
    'cloud.balanced-mini-fires.yaml',
    'cloud.default.yaml',
    'cloud.quick-screen.yaml',
    'cloud.recall.yaml',
    'cloud.robust.yaml',
    'cloud.small-fire.yaml',
    'cloud.hard-negatives.yaml',
    'cloud.capacity.yaml',
    'cloud.fast-debug.yaml',
]


USE_READY_CONFIG_AS_IS = True
READY_CONFIG_NAME = 'cloud.balanced-mini-fires.yaml'


BASE_CONFIG_NAME = 'cloud.balanced-mini-fires.yaml'

PROJECT_LABEL = 'drive-persistent'
DATASET_LABEL = 'synthetic-fire-cloud'
FIRE_IMAGE_PATHS = [
    'base_fire_images/fire1-mini_nobg.png',
    'base_fire_images/fire2-mini_nobg.png',
]
NUM_IMAGES = 2400
IMAGE_SIZE = 640
NEGATIVE_RATIO = 0.3
TRAIN_SPLIT = 0.8
DATASET_SEED = 44
FORCE_REGENERATE = False

TRAINING_LABEL = 'yolov8n-mini-balanced'
MODEL_SIZE = 'n'
WEIGHTS = 'yolov8n.pt'
DEVICE = '0'
EPOCHS = 45
BATCH_SIZE = 16
RESUME_POLICY = 'auto'

DATASET_SETTINGS_OVERRIDES = {
    'fire_scale_min': 0.04,
    'fire_scale_max': 0.52,
}
IMAGE_TRANSFORM_OVERRIDES = {
    'rotation_deg_min': -160,
    'rotation_deg_max': 160,
    'perspective_shift': 130,
    'enable_color_shift': True,
    'color_shift_prob': 0.45,
    'motion_blur_prob': 0.35,
    'gaussian_blur_prob': 0.25,
    'noise_prob': 0.25,
    'shadow_prob': 0.55,
    'occlusion_prob': 0.35,
    'augment_negative_backgrounds': True,
}
TRAINING_OVERRIDES = {
    'patience': 12,
    'learning_rate_init': 0.008,
    'learning_rate_final': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'rotation_degrees': 12,
    'translate': 0.1,
    'scale': 0.45,
    'flip_vertical': 0.4,
    'flip_horizontal': 0.5,
    'mosaic': 0.8,
    'mixed_precision': True,
    'overwrite_existing': True,
    'verbose': True,
}

MANUAL_CONFIG_PATCH: dict = {}

In [ ]:
# Patch manuale opzionale: puoi modificare QUALSIASI chiave annidata.
# Esempio: {'dataset': {'num_images': 1200}, 'training': {'epochs': 30}}
MANUAL_CONFIG_PATCH = {}


def deep_update(base: dict, patch: dict) -> dict:
    for key, value in patch.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_selected_base_config() -> tuple[dict, Path, str]:
    selected_config_name = READY_CONFIG_NAME if USE_READY_CONFIG_AS_IS else BASE_CONFIG_NAME
    config_path = REPO_ROOT / 'configs' / selected_config_name
    if not config_path.exists():
        raise FileNotFoundError(f'Config base non trovata: {config_path}')

    with open(config_path, 'r', encoding='utf-8') as handle:
        loaded = yaml.safe_load(handle) or {}
    if not isinstance(loaded, dict):
        raise ValueError(f'Config YAML non valida: {config_path}')
    return loaded, config_path, selected_config_name


def resolve_effective_device(requested_device: str | None) -> str:
    normalized = str(requested_device).strip() if requested_device is not None else 'cpu'
    if normalized.lower() == 'cpu':
        return 'cpu'
    if globals().get('CUDA_READY', False):
        return normalized
    print(f"ATTENZIONE: device richiesto '{normalized}' ma torch non vede CUDA. Fallback automatico a 'cpu'.")
    return 'cpu'


config, source_config_path, selected_config_name = load_selected_base_config()
config.setdefault('project', {})
config.setdefault('dataset', {})
config.setdefault('training', {})
config.setdefault('dataset_settings_overrides', {})
config.setdefault('image_transform_overrides', {})
config.setdefault('training_overrides', {})

config['project']['persistent_root'] = str(PERSISTENT_ROOT)

if not USE_READY_CONFIG_AS_IS:
    config['project']['label'] = PROJECT_LABEL
    config['dataset']['label'] = DATASET_LABEL
    config['dataset']['fire_image_paths'] = FIRE_IMAGE_PATHS
    config['dataset']['num_images'] = NUM_IMAGES
    config['dataset']['image_size'] = IMAGE_SIZE
    config['dataset']['negative_ratio'] = NEGATIVE_RATIO
    config['dataset']['train_split'] = TRAIN_SPLIT
    config['dataset']['seed'] = DATASET_SEED
    config['dataset']['force_regenerate'] = FORCE_REGENERATE

    config['training']['label'] = TRAINING_LABEL
    config['training']['model_size'] = MODEL_SIZE
    config['training']['weights'] = WEIGHTS
    config['training']['device'] = DEVICE
    config['training']['epochs'] = EPOCHS
    config['training']['batch_size'] = BATCH_SIZE
    config['training']['image_size'] = IMAGE_SIZE
    config['training']['resume'] = RESUME_POLICY

    config['dataset_settings_overrides'] = dict(DATASET_SETTINGS_OVERRIDES)
    config['image_transform_overrides'] = dict(IMAGE_TRANSFORM_OVERRIDES)
    config['training_overrides'] = dict(TRAINING_OVERRIDES)

requested_training_device = config.get('training', {}).get('device', 'cpu')
effective_training_device = resolve_effective_device(requested_training_device)
config['training']['device'] = effective_training_device

if MANUAL_CONFIG_PATCH:
    config = deep_update(config, MANUAL_CONFIG_PATCH)
    print('Patch manuale applicata.')
    requested_training_device = config.get('training', {}).get('device', 'cpu')
    effective_training_device = resolve_effective_device(requested_training_device)
    config['training']['device'] = effective_training_device

RUNTIME_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=False)

print('Preset/config sorgente selezionata =', selected_config_name)
print('Config sorgente letta da =', source_config_path)
print('USE_READY_CONFIG_AS_IS =', USE_READY_CONFIG_AS_IS)
print('Device richiesto =', requested_training_device)
print('Device effettivo =', effective_training_device)
print('Config finale che verra eseguita =', RUNTIME_CONFIG_PATH)
print('\nContenuto runtime config:')
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

In [ ]:
print('Verifica runtime config')
print('Preset sorgente selezionato =', selected_config_name)
print('Path runtime config =', RUNTIME_CONFIG_PATH)
print('Esiste? =', RUNTIME_CONFIG_PATH.exists())
print('Cartella configs esiste? =', RUNTIME_CONFIG_PATH.parent.exists())
print('Contenuto cartella configs:')
for path in sorted(RUNTIME_CONFIG_PATH.parent.glob('*')):
    print('-', path.name)

if RUNTIME_CONFIG_PATH.exists():
    print('\nPrime righe della runtime config:')
    print('\n'.join(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8').splitlines()[:40]))
else:
    raise FileNotFoundError(
        f'Runtime config non generata in {RUNTIME_CONFIG_PATH}. Riesegui la cella 9 e controlla eventuali errori sopra.'
    )

In [ ]:
# Optional local smoke override for notebook validation
if not IN_COLAB:
    with open(RUNTIME_CONFIG_PATH, 'r', encoding='utf-8') as handle:
        smoke_config = yaml.safe_load(handle)

    smoke_config['project']['label'] = 'cloud-notebook-smoke'
    smoke_config['dataset']['label'] = 'cloud-notebook-smoke-dataset'
    smoke_config['dataset']['num_images'] = 12
    smoke_config['dataset']['image_size'] = 320
    smoke_config['dataset']['force_regenerate'] = True
    smoke_config['training']['label'] = 'cloud-notebook-smoke'
    smoke_config['training']['model_size'] = 'n'
    smoke_config['training']['weights'] = 'yolov8n.pt'
    smoke_config['training']['epochs'] = 1
    smoke_config['training']['batch_size'] = 4
    smoke_config['training']['image_size'] = 320
    smoke_config['training']['device'] = 'cpu'
    smoke_config['training']['resume'] = 'never'

    with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as handle:
        yaml.safe_dump(smoke_config, handle, sort_keys=False, allow_unicode=False)

    print('Local notebook smoke override applied:')
    print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

## Esecuzione della pipeline

La cella seguente esegue l'entrypoint ufficiale del progetto:

```bash
python run_experiment.py --config configs/cloud.runtime.yaml
```

Questo e' corretto anche se sopra hai selezionato un preset come `cloud.balanced-mini-fires.yaml`. Il notebook non passa direttamente il preset al training:

1. legge il preset sorgente scelto
2. costruisce una config finale aggiornata
3. salva quella config finale in `configs/cloud.runtime.yaml`
4. lancia `run_experiment.py` usando proprio `cloud.runtime.yaml`

Quindi:

- `cloud.balanced-mini-fires.yaml` e' la config sorgente selezionata
- `cloud.runtime.yaml` e' la config effettivamente eseguita

Questo comando orchestra tutto il flusso:

- riuso o generazione del dataset sintetico
- training YOLOv8
- registrazione dell'export finale
- aggiornamento del registry in `exports/latest.yaml`

Qui il comando viene lanciato direttamente dal notebook, senza wrapper `subprocess`, per vedere l'output del training in modo piu' fedele possibile.

Non serve eseguire moduli interni come `generator.py` o `train.py` separatamente.

In [ ]:
if not RUNTIME_CONFIG_PATH.exists():
    raise FileNotFoundError(
        'Runtime config non trovata. Esegui prima la cella di configurazione che salva configs/cloud.runtime.yaml.'
    )

get_ipython().run_line_magic('cd', str(REPO_ROOT))
get_ipython().system(f'"{sys.executable}" -u run_experiment.py --config "{RUNTIME_CONFIG_PATH}"')

## Output persistenti e download su PC

Dopo l'esecuzione della pipeline, tutti i risultati restano sotto la stessa root persistente.

Di default, in Colab, la root e':

```text
/content/drive/MyDrive/yolo-fire-detector
```

### `datasets/`

Ogni dataset sta in:

```text
datasets/<dataset-label>-<fingerprint>/
```

Dentro trovi:

- `yolo_dataset.yaml`
- `dataset_manifest.yaml`
- `images/`
- `labels/`

### `runs/`

Ogni run sta in:

```text
runs/<run_label>/
```

Dentro trovi almeno:

- `resolved_config.yaml`
- `pipeline_summary.yaml`
- `training_run.yaml`
- metriche e diagnostica YOLO

Durante il training possono comparire checkpoint in `weights/`. Se la run termina con successo, i checkpoint di resume vengono rimossi automaticamente.

### `exports/`

Qui trovi gli artefatti finali per inferenza:

- `exports/<run_label>.pt`
- `exports/<run_label>.yaml`
- `exports/latest.yaml`

`latest.yaml` punta all'export corrente.

### Download sul PC locale

L'ultima cella del notebook legge `exports/latest.yaml`, comprime tutta la cartella `exports/` in `downloads/yolo-fire-detector-exports.zip` e, in Colab, stampa anche il comando per scaricare lo zip sul PC locale.

### Se riapri una sessione

Riesegui semplicemente il notebook:

- il dataset viene riusato se il fingerprint coincide
- la run puo' riprendere se esiste un checkpoint compatibile e `training.resume` lo consente

In [ ]:
latest_registry = PERSISTENT_ROOT / 'exports' / 'latest.yaml'
exports_root = PERSISTENT_ROOT / 'exports'
download_root = PERSISTENT_ROOT / 'downloads'
download_root.mkdir(parents=True, exist_ok=True)
exports_archive = download_root / 'yolo-fire-detector-exports.zip'

if latest_registry.exists():

    print('Latest export registry:')
    print(latest_registry.read_text(encoding='utf-8'))

else:

    print('Nessun export registrato ancora.')


print('\nExports disponibili:')

for path in sorted(exports_root.glob('*')):

    print('-', path)


if exports_root.exists():
    if exports_archive.exists():
        exports_archive.unlink()
    shutil.make_archive(str(exports_archive.with_suffix('')), 'zip', root_dir=exports_root, base_dir='.')
    print('\nArchivio export pronto:', exports_archive)

    if IN_COLAB:
        print('\nPer salvare tutto sul PC locale:')
        print('from google.colab import files')
        print(f"files.download('{exports_archive}')")

print('\nDatasets disponibili:')

for path in sorted((PERSISTENT_ROOT / 'datasets').glob('*')):

    print('-', path)


print('\nRun disponibili:')

for path in sorted((PERSISTENT_ROOT / 'runs').glob('*')):

    print('-', path)